In [3]:
from pathlib import Path
import sys
import numpy as np
import tensorflow as tf

sys.path.append(str(Path("../src").resolve()))

from config import (
    FINAL_17_CHANNELS,
    WINDOW_SIZE_SEC,
    TARGET_SFREQ
)

from utils import (
    get_final17_prediction_file_lists,
    prediction_batch_generator,
    prediction_eval_generator,
    build_dataset_balance_plan,
    print_dataset_balance_plan,
     transpose_generator
     
)

2026-05-01 18:57:42.587956: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 18:57:42.721091: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-01 18:57:45.152956: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# This function builds the baseline CNN model for seizure prediction
# The model will learn to classify:
# 0 = interictal (normal brain activity)
# 1 = preictal (before seizure)

def build_baseline_cnn(input_shape):

    # Sequential means we stack layers one after another
    model = tf.keras.Sequential([

        # Input layer
        # input_shape = (timepoints, channels)
        # Example: (1280, 17)
        tf.keras.layers.Input(shape=input_shape),

        
        # This is the first layer that looks at the EEG signal
        # It tries to detect small/simple patterns like:
        # small spikes,short oscillations and tiny waveform changes
        
        tf.keras.layers.Conv1D(
            filters=32,          # number of filters (features the model will learn)
            kernel_size=7,       # size of the window scanning the signal
            activation="relu",  # adds non-linearity so model can learn complex patterns
            padding="same"      # keeps output size same as input
        ),

        # BatchNormalization makes training more stable
        # it normalizes the output so values are easier to learn from
        tf.keras.layers.BatchNormalization(),

        # MaxPooling reduces the size
        # keeps important info but removes unnecessary details
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # Dropout randomly disables neurons during training
        # helps prevent overfitting (model memorizing instead of learning)
        tf.keras.layers.Dropout(0.25),


    
        # This block takes features from Block 1 and combines them
        # It starts learning stronger patterns like repeated spikes and abnormal rhythms
        
        tf.keras.layers.Conv1D(
            filters=64,
            kernel_size=5,
            activation="relu",
            padding="same"
        ),

        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Dropout(0.25),


        # This block learns higher-level patterns
        # These are more meaningful for prediction like preictal signatures and patterns before seizure starts

        tf.keras.layers.Conv1D(
            filters=128,
            kernel_size=3,
            activation="relu", #used the relu for non leanrity and to avoid vanishing gradient
            padding="same"
        ),

        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Dropout(0.30),


        # This layer converts all feature maps into one vector
        # It reduces the data size and keeps the most important information
        tf.keras.layers.GlobalAveragePooling1D(),


        # Dense layer combines all learned features
        # This is where the model makes sense of everything it learned
        tf.keras.layers.Dense(64, activation="relu"),

        # More dropout to reduce overfitting before final prediction
        tf.keras.layers.Dropout(0.40),


        # Final output layer
        # 1 neuron because this is binary classification
        # sigmoid gives a value between 0 and 1
        # closer to 1 = more likely preictal
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])


    
    model.compile(

        # Adam optimizer adjusts weights during training
        # learning_rate controls how fast the model learns
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),

        
        loss="binary_crossentropy",

        
        metrics=[
            "accuracy",  # overall correct predictions

            # precision = how many predicted preictal are actually correct
            tf.keras.metrics.Precision(name="precision"),

            # recall = how many real preictal cases we detected
            tf.keras.metrics.Recall(name="recall"),

            # this measures how well model separates classes
            tf.keras.metrics.AUC(name="auc")
        ]
    )


    return model

In [ ]:
import json
from pathlib import Path
import numpy as np

from utils import (
    get_final17_prediction_file_lists,
    prediction_batch_generator,
    transpose_generator
)

# 1. Load file lists
train_files, val_files, test_files = get_final17_prediction_file_lists()

print("Total train files:", len(train_files))

# 2. Load saved balance plan instead of scanning again
balance_plan_path = Path("../results/balance_plan_train_ratio_4.json")

with open(balance_plan_path, "r") as f:
    balance_plan = json.load(f)

print("Loaded balance plan:")
print(balance_plan)


# NEW - use full training set
train_files_full = train_files

print("Full training files:", len(train_files_full))

# 4. Create generator
batch_size = 32

train_generator = prediction_batch_generator(
    file_paths=train_files_full,
    batch_size=batch_size,
    balance_plan=balance_plan,
    preictal_fraction=0.25,
    shuffle=True,
    random_state=42
)

# 5. Transpose for CNN
train_generator = transpose_generator(train_generator)

# 6. Check one batch
X_batch, y_batch = next(train_generator)

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)

print("Preictal:", np.sum(y_batch == 1))
print("Interictal:", np.sum(y_batch == 0))

print("NaN:", np.isnan(X_batch).any())
print("Inf:", np.isinf(X_batch).any())
print("Range:", X_batch.min(), X_batch.max())

Total train files: 455
Loaded balance plan:
{'total_preictal': 1929, 'total_interictal': 1069490, 'target_interictal': 7716, 'keep_probability': 0.007214653713452206, 'train_ratio': 4}
Full training files: 455


In [9]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path("/mnt/c/Users/MSI/Desktop/EEG_FYP")
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from utils import get_final17_prediction_file_lists
from config import WINDOW_SIZE_SEC

train_files, val_files, test_files = get_final17_prediction_file_lists()

print("Checking validation durations...\n")

for file_path in val_files:
    data = np.load(file_path)

    X = data["X"]  # shape: (num_windows, timepoints, channels)

    num_windows = X.shape[0]

    # each window = 5 seconds (your config)
    duration_sec = num_windows * WINDOW_SIZE_SEC
    duration_min = duration_sec / 60

    print(file_path.name, "| windows:", num_windows, "| duration:", round(duration_min, 2), "min")

Checking validation durations...

chb03_01_processed.npz | windows: 1439 | duration: 119.92 min
chb03_02_processed.npz | windows: 1439 | duration: 119.92 min
chb03_03_processed.npz | windows: 1439 | duration: 119.92 min
chb03_04_processed.npz | windows: 1439 | duration: 119.92 min
chb03_05_processed.npz | windows: 396 | duration: 33.0 min
chb03_06_processed.npz | windows: 1439 | duration: 119.92 min
chb03_07_processed.npz | windows: 52 | duration: 4.33 min
chb03_08_processed.npz | windows: 1439 | duration: 119.92 min


KeyboardInterrupt: 

In [13]:
summary_path = RAW_ROOT / "chb03" / "chb03-summary.txt"

with open(summary_path, "r") as f:
    lines = f.readlines()

for line in lines:
    if "File Name:" in line or "Seizure" in line:
        print(line.strip())

In [4]:
import importlib
import utils
importlib.reload(utils)

<module 'utils' from '/mnt/c/Users/MSI/Desktop/EEG_FYP/src/utils.py'>

In [10]:
from config import RAW_ROOT
from pathlib import Path
import re

summary_path = RAW_ROOT / "chb20" / "chb20-summary.txt"

print("Path:", summary_path)
print("Exists:", summary_path.exists())

with open(summary_path, "r") as f:
    lines = f.readlines()

print("Total lines:", len(lines))

for line in lines:
    if "File Name" in line or "Seizure" in line:
        print(repr(line))

import re

current_file = None
seizure_info = {}
starts = []
ends = []

for line in lines:
    line = line.strip()

    if line.startswith("File Name:"):
        if current_file is not None:
            seizure_info[current_file] = [
                {"start": s, "end": e}
                for s, e in zip(starts, ends)
            ]

        current_file = line.replace("File Name:", "").strip()
        starts = []
        ends = []

    elif "Seizure" in line and "Start Time" in line:
        match = re.search(r"(\d+)\s+seconds", line)
        print("START MATCH:", line, match.group(1) if match else None)
        if match:
            starts.append(int(match.group(1)))

    elif "Seizure" in line and "End Time" in line:
        match = re.search(r"(\d+)\s+seconds", line)
        print("END MATCH:", line, match.group(1) if match else None)
        if match:
            ends.append(int(match.group(1)))

if current_file is not None:
    seizure_info[current_file] = [
        {"start": s, "end": e}
        for s, e in zip(starts, ends)
    ]

print("\nFILES WITH SEIZURES:")
for file_name, seizures in seizure_info.items():
    if len(seizures) > 0:
        print(file_name, seizures)

Path: /mnt/c/Users/MSI/Desktop/EEG_FYP/data/raw/chb20/chb20-summary.txt
Exists: True
Total lines: 194
'File Name: chb20_01.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_02.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_03.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_04.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_05.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_06.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_07.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_08.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_11.edf\n'
'Number of Seizures in File: 0\n'
'File Name: chb20_12.edf\n'
'Number of Seizures in File: 1\n'
'Seizure 1 Start Time: 94 seconds\n'
'Seizure 1 End Time: 123 seconds\n'
'File Name: chb20_13.edf\n'
'Number of Seizures in File: 2\n'
'Seizure 1 Start Time: 1440 seconds\n'
'Seizure 1 End Time: 1470 seconds\n'
'Seizure 2 Start Time: 2498 seconds\n'
'Seizure 2 End Time: 2537 se

In [11]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path("/mnt/c/Users/MSI/Desktop/EEG_FYP")
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from utils import (
    get_final17_prediction_file_lists,
    load_prediction_labels_only
)

train_files, val_files, test_files = get_final17_prediction_file_lists()

print("Validation files:", len(val_files))

total_preictal = 0
total_interictal = 0

print("\nValidation files that contain preictal windows:")
print("=" * 60)

for file_path in val_files:
    y = load_prediction_labels_only(file_path)

    preictal = int(np.sum(y == 1))
    interictal = int(np.sum(y == 0))

    total_preictal += preictal
    total_interictal += interictal

    if preictal > 0:
        print(file_path.name, "| preictal:", preictal, "| interictal:", interictal)

print("=" * 60)
print("TOTAL validation preictal:", total_preictal)
print("TOTAL validation interictal:", total_interictal)

if total_preictal > 0:
    print("YES validation has preictal windows")
    print("Interictal / Preictal ratio:", total_interictal / total_preictal)
else:
    print("NO validation still has 0 preictal windows")

Validation files: 67

Validation files that contain preictal windows:
chb03_01_processed.npz | preictal: 23 | interictal: 1273
chb03_02_processed.npz | preictal: 171 | interictal: 1120
chb03_03_processed.npz | preictal: 51 | interictal: 1238
chb03_04_processed.npz | preictal: 240 | interictal: 1056
chb03_34_processed.npz | preictal: 240 | interictal: 1058
chb03_35_processed.npz | preictal: 240 | interictal: 1051
chb03_36_processed.npz | preictal: 240 | interictal: 1056
chb20_13_processed.npz | preictal: 480 | interictal: 689
chb20_14_processed.npz | preictal: 240 | interictal: 1062
chb20_15_processed.npz | preictal: 275 | interictal: 887
chb20_16_processed.npz | preictal: 240 | interictal: 1010
chb20_68_processed.npz | preictal: 240 | interictal: 488
TOTAL validation preictal: 2680
TOTAL validation interictal: 87265
YES validation has preictal windows
Interictal / Preictal ratio: 32.5615671641791


In [ ]:
from pathlib import Path
import tensorflow as tf

# 1. Build the FULL model
model = build_baseline_cnn(input_shape=(1280, 17))

# 2. Check that BatchNormalization + Dropout are included
model.summary()

# 3. Prepare save folders
models_dir = Path("../models")
results_dir = Path("../results")

models_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

# 4. Callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=models_dir / "full_baseline_cnn_best.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        filename=results_dir / "full_baseline_cnn_training_log.csv"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

# 5. Train full model on the whole training generator
history = model.fit(
    train_generator,
    steps_per_epoch=300,
    epochs=30,
    callbacks=callbacks
)

# 6. Save final model
final_model_path = models_dir / "full_baseline_cnn_prediction_final.keras"
model.save(final_model_path)

print("Final full model saved to:", final_model_path)

I0000 00:00:1777553529.131539   55904 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1274, 32)       │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 637, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 633, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 316, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 314, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,169 (184.25 KB)

 Trainable params: 47,169 (184.25 KB)

...

Epoch 1/10


2026-04-30 14:52:12.573269: I external/local_xla/xla/service/service.cc:163] XLA service 0x791d8800c640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-30 14:52:12.573304: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-04-30 14:52:12.661409: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-30 14:52:13.259920: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92101
2026-04-30 14:52:13.415520: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-04-30 14:52:13.

  6/300 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.6073 - auc: 0.4502 - loss: 0.6654 - precision: 0.2538 - recall: 0.3063

I0000 00:00:1777553537.779003   59142 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


140/300 ━━━━━━━━━━━━━━━━━━━━ 31:40 12s/step - accuracy: 0.7411 - auc: 0.5835 - loss: 0.5558 - precision: 0.3707 - recall: 0.0657

In [ ]:
"A total of ~9000 balanced EEG windows were used for training, consisting of approximately 2250 preictal and 6750 interictal samples, maintaining a 3:1 ratio. Data was generated using patient-wise splits to ensure generalisation, and batch caching was implemented to improve training efficiency."

# The Below Code Will be used for Batches Loading

In [ ]:
import json
import time
from pathlib import Path
import numpy as np

from utils import (
    prediction_batch_generator,
    transpose_generator,
    get_final17_prediction_file_lists
)

# folders
cache_dir = Path("../data/cached_train_batches")
cache_dir.mkdir(parents=True, exist_ok=True)

results_dir = Path("../results")

#  use real train files
train_files, val_files, test_files = get_final17_prediction_file_lists()
print("Train files:", len(train_files))

# load saved balance plan
with open(results_dir / "balance_plan_train_ratio_4.json", "r") as f:
    balance_plan = json.load(f)

# settings
batch_size = 8
num_batches_to_cache = 500

# resume logic
existing_batches = sorted(cache_dir.glob("train_batch_*.npz"))
start_index = len(existing_batches)

print("Existing cached batches:", start_index)
print("New batches to add:", num_batches_to_cache)

# generator
train_generator = prediction_batch_generator(
    file_paths=train_files,
    batch_size=batch_size,
    balance_plan=balance_plan,
    preictal_fraction=0.25,
    shuffle=True,
    random_state=42 + start_index
)

train_generator = transpose_generator(train_generator)

print("Creating cached batches...")
start_time = time.time()

for i in range(num_batches_to_cache):
    batch_index = start_index + i

    X_batch, y_batch = next(train_generator)

    save_path = cache_dir / f"train_batch_{batch_index:05d}.npz"

    np.savez(
        save_path,
        X=X_batch.astype(np.float32),
        y=y_batch.astype(np.int64)
    )

    if i % 20 == 0:
        print(f"[{i}/{num_batches_to_cache}] saved")

print("DONE")
print("Total batches now:", len(list(cache_dir.glob("train_batch_*.npz"))))
print(f"Time: {(time.time() - start_time)/60:.2f} min")

Train files: 455
Existing cached batches: 240
New batches to add: 500
Creating cached batches...
Loaded patient channel map from: /mnt/d/EEG_FYP_DATA/processed/patient_channel_map.json


KeyboardInterrupt: 

In [ ]:
import numpy as np
import random
from pathlib import Path

def cached_batch_generator(cache_dir, shuffle=True):
    cache_dir = Path(cache_dir)

    batch_files = sorted(cache_dir.glob("train_batch_*.npz"))

    if len(batch_files) == 0:
        raise ValueError("No cached batch files found.")

    while True:
        files = batch_files.copy()

        if shuffle:
            random.shuffle(files)

        for file_path in files:
            data = np.load(file_path)

            X = data["X"]
            y = data["y"]

            yield X, y

In [9]:
# check class balance in batch
from config import PROCESSED_ROOT, PROCESSED_GROUP1, PROCESSED_GROUP2, PROCESSED_GROUP3, PROCESSED_GROUP4

print(PROCESSED_ROOT)
print("Root exists:", PROCESSED_ROOT.exists())

print("Group1 exists:", PROCESSED_GROUP1.exists())
print("Group2 exists:", PROCESSED_GROUP2.exists())
print("Group3 exists:", PROCESSED_GROUP3.exists())
print("Group4 exists:", PROCESSED_GROUP4.exists())

/mnt/d/EEG_FYP_DATA/processed
Root exists: True
Group1 exists: True
Group2 exists: True
Group3 exists: True
Group4 exists: True


In [10]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs found:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.20.0
GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
val_generator = prediction_eval_generator(
    file_paths=val_files,
    batch_size=32
)

val_generator = transpose_generator(val_generator)

history = model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=10,
    validation_data=val_generator,
    validation_steps=30
)

In [ ]:


Path("../models").mkdir(exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="../models/baseline_cnn_prediction_best.keras",
        monitor="val_auc",# Best AUC for validation
        mode="max",
        save_best_only=True,# onily save when the model improves 
        verbose=1 #print message when saving
    ),
#stops when the model is not improving
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
#reduces learning rate when the model is not improving
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5, #if ite gets stuck reducing by half
        patience=3, # witing 3 epochs before reducing
        min_lr=1e-6,
        verbose=1
    )
]

val_generator = prediction_eval_generator(
    file_paths=val_files,
    batch_size=32
)

val_generator = transpose_generator(val_generator)

history = model.fit(
    train_generator,
    steps_per_epoch=300,
    epochs=30,
    validation_data=val_generator,
    validation_steps=80,
    callbacks=callbacks
)

Epoch 1/30
299/300 ━━━━━━━━━━━━━━━━━━━━ 0s 806ms/step - accuracy: 0.9266 - auc: 0.9578 - loss: 0.2070 - precision: 0.8951 - recall: 0.8014

2026-04-26 18:07:52.183970: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-04-26 18:07:53.010203: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_207', 28 bytes spill stores, 28 bytes spill loads




Epoch 1: val_auc improved from None to 0.00000, saving model to ../models/baseline_cnn_prediction_best.keras

Epoch 1: finished saving model to ../models/baseline_cnn_prediction_best.keras
300/300 ━━━━━━━━━━━━━━━━━━━━ 262s 875ms/step - accuracy: 0.9152 - auc: 0.9386 - loss: 0.2541 - precision: 0.8582 - recall: 0.7917 - val_accuracy: 0.9226 - val_auc: 0.0000e+00 - val_loss: 0.1821 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 5.0000e-04
Epoch 2/30
298/300 ━━━━━━━━━━━━━━━━━━━━ 1s 728ms/step - accuracy: 0.9112 - auc: 0.9450 - loss: 0.2296 - precision: 0.8478 - recall: 0.7888
Epoch 2: val_auc did not improve from 0.00000
300/300 ━━━━━━━━━━━━━━━━━━━━ 235s 785ms/step - accuracy: 0.9116 - auc: 0.9373 - loss: 0.2476 - precision: 0.8774 - recall: 0.7513 - val_accuracy: 1.0000 - val_auc: 0.0000e+00 - val_loss: 0.0144 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 5.0000e-04
Epoch 3/30
152/300 ━━━━━━━━━━━━━━━━━━━━ 1:31 616ms/step - accuracy: 0.9434 